In [6]:
import numpy as np
import pandas as pd
import os


RESULTS_PATH = "/home/leya/Desktop/results"
PROPERTIES_PATH = "data/target_properties_v2.csv"

In [7]:
# files_list = os.listdir(RESULTS_PATH)

In [8]:
# data = pd.read_json(f"{RESULTS_PATH}/{files_list[0]}",lines=True)

In [9]:
# data = data[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

In [10]:
# for f in files_list[1:]:
#     temp = pd.read_json(f"{RESULTS_PATH}/{f}",lines=True)
#     temp = temp[['targetid', 'rew_civ', 'rew_nv', 'rew_lya', 'snr_1700A', 'snr_civ', 'snr_nv', 'snr_lya', 'pixel_used_civ', 'pixel_used_blue_end_lya', 'continuum_redchi']].copy()

#     data = pd.concat([data,temp])

In [11]:
# data.to_csv("data/local/first_results.csv")
data = pd.read_csv("data/local/first_results.csv",index_col=0)
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 199567 entries, 0 to 18000
Data columns (total 11 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   targetid                 199567 non-null  int64  
 1   rew_civ                  199533 non-null  object 
 2   rew_nv                   196748 non-null  float64
 3   rew_lya                  199562 non-null  float64
 4   snr_1700A                199563 non-null  float64
 5   snr_civ                  199566 non-null  float64
 6   snr_nv                   199565 non-null  float64
 7   snr_lya                  199565 non-null  float64
 8   pixel_used_civ           199566 non-null  float64
 9   pixel_used_blue_end_lya  199566 non-null  float64
 10  continuum_redchi         199566 non-null  float64
dtypes: float64(9), int64(1), object(1)
memory usage: 18.3+ MB


/tmp/ipykernel_41109/2281834570.py:2: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("data/local/first_results.csv",index_col=0)


In [12]:
data = data[(data["rew_civ"] != "continuum fit failed")]
data["rew_civ"] = data["rew_civ"].astype("float")

In [13]:
data["quality_flag"] = (data["snr_civ"] < 2.5) | (data["continuum_redchi"] > 2) | (data["rew_civ"] == "continuum fit failed")

data["flag_civ"] = data["rew_civ"].isna() | (data["pixel_used_civ"] < 0.7) | data["quality_flag"]
data["flag_nv"] = data["rew_nv"].isna() | data["rew_lya"].isna() | data["quality_flag"]
data["flag_lya"] = data["rew_lya"].isna() | data["rew_nv"].isna() | (data["pixel_used_blue_end_lya"] < 0.65) | data["quality_flag"]

In [14]:
data[data["flag_civ"]==False]["rew_civ"].describe()

count    1.103040e+05
mean     2.651172e+10
std      8.805083e+12
min     -1.041377e+01
25%      3.592360e+01
50%      5.252366e+01
75%      8.044755e+01
max      2.924348e+15
Name: rew_civ, dtype: float64

In [15]:
data[(data["rew_civ"] > 100) & (data["flag_civ"]==False)].sort_values("rew_civ")

,targetid,rew_civ,rew_nv,rew_lya,snr_1700A,snr_civ,snr_nv,snr_lya,pixel_used_civ,pixel_used_blue_end_lya,continuum_redchi,quality_flag,flag_civ,flag_nv,flag_lya
7470,39627738520227832,1.000039e+02,23.898937,36.411235,0.817014,3.698828,1.415084,1.819015,0.966265,0.884718,0.025275,False,False,False,False
6685,39627709655023752,1.000055e+02,7.132390,44.136226,0.670148,4.261444,0.790222,3.213076,0.990050,0.850416,0.042893,False,False,False,False
7052,39627720874791032,1.000074e+02,13.329574,12.183155,0.516218,5.637394,2.711289,3.563045,0.953846,0.753056,0.038162,False,False,False,False
17492,39627978077901440,1.000075e+02,38.518246,114.041022,0.727549,10.046167,3.989132,12.675547,0.973094,0.855362,0.066825,False,False,False,False
16579,39627990136522168,1.000097e+02,51.148255,58.819386,0.745425,3.466310,1.790865,3.782243,0.949187,0.854260,0.064671,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230,39627218237784232,2.427195e+03,220.618873,400.737548,0.631899,3.433029,2.036620,4.128201,0.959924,0.842553,0.069136,False,False,False,False
405,39627739598164360,2.547973e+03,0.000000,0.000000,0.501755,3.751734,1.100318,2.943573,0.928571,0.871233,0.025049,False,False,False,False
15651,39627923530976736,2.626252e+03,78.601580,571.260263,0.665555,3.017073,1.404251,7.699725,0.913043,0.875274,0.016050,False,False,False,False
1400,39627770141086328,3.954097e+03,77.933054,336.183526,0.747524,2.746914,0.984409,2.319498,0.916268,0.868633,0.012238,False,False,False,False


In [16]:
properties = pd.read_csv(PROPERTIES_PATH,index_col=0)
properties

,targetid,z,zerr,civ_1549_flux,civ_1549_flux_ivar,flux_z,flux_ivar_z,flux_w3,flux_ivar_w3,desiname
0,39628176690778063,3.362767,0.000090,217.312470,0.105604,1.705293,45.407578,19.646732,0.001212,DESI J256.8974+16.1830
1,39628285633632556,2.977364,0.000637,69.612305,0.162742,2.176984,152.734680,98.437126,0.001273,DESI J262.5817+21.0899
2,39628308123487220,2.482214,0.000215,89.079796,0.242858,5.418437,93.714360,91.612335,0.001438,DESI J261.3568+22.0109
3,39628280034233213,2.296296,0.000738,41.549404,0.249651,7.905924,231.985150,23.411120,0.001280,DESI J265.5436+20.8310
4,39628205652448265,3.135919,0.000115,419.981700,0.041608,17.866806,129.960000,96.393880,0.001995,DESI J259.1587+17.4301
...,...,...,...,...,...,...,...,...,...,...
404137,39628335873004003,2.530704,0.000855,98.225820,0.048998,2.058528,63.326935,-72.672300,0.001359,DESI J251.3572+23.1657
404138,39628234161127763,2.250925,0.000381,212.045300,0.025861,8.608725,52.475870,117.521904,0.001161,DESI J244.3208+18.6837
404139,39628368961868916,3.172024,0.000298,165.221860,0.155956,3.854564,81.393234,42.885216,0.001127,DESI J247.5476+24.6466
404140,39628285482633552,2.717202,0.000902,22.965364,0.075073,0.691328,80.447510,-9.402734,0.001025,DESI J252.8584+20.9717


In [26]:
temp = data[(data["rew_civ"] > 100) & (data["flag_civ"]==False)].copy()
merged = temp.merge(properties,how="inner",on="targetid")

In [31]:
merged["mag_diff"] = merged["flux_z"] - merged["flux_w3"]
merged[merged["mag_diff"]>3.9]

,targetid,rew_civ,rew_nv,rew_lya,snr_1700A,snr_civ,snr_nv,snr_lya,pixel_used_civ,pixel_used_blue_end_lya,...,z,zerr,civ_1549_flux,civ_1549_flux_ivar,flux_z,flux_ivar_z,flux_w3,flux_ivar_w3,desiname,mag_diff
0,39627739203896616,125.386553,52.129702,39.393725,0.603982,4.630902,1.271301,4.791493,0.993435,0.839416,...,2.656260,0.000350,94.18471,0.137303,1.684116,138.20953,-14.178690,0.000859,DESI J166.5509-01.9888,15.862806
2,39627739531054856,165.557439,54.340998,118.395239,0.728054,9.906072,3.348963,8.970880,0.966736,0.886311,...,2.844215,0.000133,209.26695,0.186948,2.018563,115.25789,-28.701800,0.000682,DESI J186.1699-01.9477,30.720363
5,39627739820462864,169.835684,102.442432,164.732124,0.810209,3.456716,0.987901,3.079539,0.938356,0.911168,...,2.508983,0.000409,124.75333,0.082218,1.390830,99.35907,-28.072416,0.001350,DESI J203.4851-02.0085,29.463246
6,39627740135031120,113.902708,48.563094,58.630074,0.680833,6.318064,2.054088,6.727376,0.978188,0.732710,...,3.768287,0.000224,32.10090,1.038054,1.065103,90.92260,-10.090407,0.001696,DESI J222.0111-02.0421,11.155510
8,39627767465118960,134.107747,58.823986,177.646488,0.834716,3.784307,1.672480,4.213040,0.930328,0.874429,...,2.904447,0.000345,42.72471,0.363868,0.706240,249.08145,-34.755653,0.000855,DESI J051.1545-00.6259,35.461893
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2227,39628024903108280,146.742966,40.362098,80.016293,0.784127,4.911870,1.845276,4.007067,0.985185,0.890110,...,2.238709,0.000408,160.11351,0.076916,1.337495,111.07231,-47.774975,0.000857,DESI J334.7257+09.8387,49.112470
2228,39628025309956152,115.242241,39.743472,53.447686,0.562218,4.091156,1.692465,3.575395,0.970660,0.855586,...,2.273959,0.000202,149.24246,0.059365,2.011691,146.35347,-8.968589,0.000798,DESI J359.3449+09.7012,10.980280
2230,39628025733580752,103.661472,62.765688,72.036880,0.641224,10.810157,8.134570,12.899545,1.000000,0.859606,...,2.615274,0.000172,339.76260,0.104821,4.045370,61.78000,-5.189320,0.001158,DESI J024.9503+09.9356,9.234690
2232,39628025741971488,185.502780,75.034590,142.662098,0.624236,5.821594,1.739648,4.203241,0.955446,0.903581,...,2.233761,0.000211,152.52599,0.102695,1.879228,124.38151,-44.558025,0.001279,DESI J025.5684+09.9698,46.437253
